# High-Fidelity DFN Simulation via Co-Simulation

The **Doyle–Fuller–Newman (DFN)** model produces a DAE system after discretisation
and cannot be used with the monolithic ODE blocks. The co-simulation blocks
(`CellCoSimElectrothermal`, `CellCoSimElectrical`) let PyBaMM step the DAE internally
while PathSim receives zero-order-held outputs between macro-steps.


## Why Co-Simulation?

The DFN model resolves spatial gradients in both the solid and electrolyte phases.
The phase-potential equations are **algebraic**, so after spatial discretisation the
DFN becomes a DAE — incompatible with `CellElectrical` / `CellElectrothermal`.

The co-simulation blocks call `pybamm.Simulation.step()` internally on a fixed macro-step
`dt` and expose zero-order-held outputs to PathSim. This supports any PyBaMM model.

> Brosa Planella et al., [arXiv:2203.16091](https://arxiv.org/abs/2203.16091) (2022).


In [ ]:
import matplotlib.pyplot as plt
import pybamm

from pathsim import Simulation, Connection
from pathsim.blocks import Constant, Scope
from pathsim.solvers import ESDIRK43

from pathsim_batt import (
    CellElectrothermal,
    CellCoSimElectrothermal,
    CellCoSimElectrical,
    LumpedThermal,
)

## Part 1: SPMe vs DFN with Built-in Thermal

Both models use PyBaMM's lumped thermal sub-model. SPMe runs as a monolithic ODE in PathSim;
DFN runs via co-simulation with a 10 s macro-step.


In [ ]:
C_nom  = 5.0     # Chen2020 nominal capacity [Ah]
I_1C   = 1.0 * C_nom  # 1 C current [A]
T_amb0 = 298.15  # ambient temperature [K]
t_end  = 1800.0  # simulation duration [s] — 50 % SoC at 1 C
dt_sim = 10.0    # PathSim macro-step [s]


def run_electrothermal(cell_block, dt_ps=10.0):
    """Helper: discharge cell_block at 1 C and return (t, V, T, SOC)."""
    I_src = Constant(I_1C)
    T_src = Constant(T_amb0)
    sco   = Scope(labels=["V", "T", "SOC"])

    sim = Simulation(
        blocks=[I_src, T_src, cell_block, sco],
        connections=[
            Connection(I_src,               cell_block["I"]),
            Connection(T_src,               cell_block["T_amb"]),
            Connection(cell_block["V"],     sco[0]),
            Connection(cell_block["T"],     sco[1]),
            Connection(cell_block["SOC"],   sco[2]),
        ],
        dt=dt_ps,
        Solver=ESDIRK43,
    )
    sim.run(t_end)
    t, [V, T, SOC] = sco.read()
    return t, V, T, SOC

In [ ]:
print("Running SPMe (monolithic)...")
spme_cell = CellElectrothermal(initial_soc=1.0)
t_spme, V_spme, T_spme, SOC_spme = run_electrothermal(spme_cell, dt_ps=dt_sim)


In [ ]:
print("Running DFN (co-simulation)...")
dfn_cell = CellCoSimElectrothermal(
    model=pybamm.lithium_ion.DFN(options={"thermal": "lumped"}),
    initial_soc=1.0,
    dt=dt_sim,
)
t_dfn, V_dfn, T_dfn, SOC_dfn = run_electrothermal(dfn_cell, dt_ps=dt_sim)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(t_spme / 3600,      V_spme,               label="SPMe",  color="steelblue")
axes[0].plot(t_dfn[1:] / 3600,  V_dfn[1:],   "--",    label="DFN",   color="orangered")
axes[0].set_xlabel("Time / h")
axes[0].set_ylabel("Terminal voltage / V")
axes[0].set_ylim(3.0, 4.3)
axes[0].legend()
axes[0].set_title("Voltage")

axes[1].plot(t_spme / 3600,     T_spme - 273.15,              label="SPMe",  color="steelblue")
axes[1].plot(t_dfn[1:] / 3600, T_dfn[1:] - 273.15, "--",     label="DFN",   color="orangered")
axes[1].set_xlabel("Time / h")
axes[1].set_ylabel("Cell temperature / °C")
axes[1].legend()
axes[1].set_title("Temperature")

axes[2].plot(t_spme / 3600,     SOC_spme * 100,              label="SPMe",  color="steelblue")
axes[2].plot(t_dfn[1:] / 3600, SOC_dfn[1:]  * 100, "--",    label="DFN",   color="orangered")
axes[2].set_xlabel("Time / h")
axes[2].set_ylabel("SOC / %")
axes[2].set_ylim(0, 105)
axes[2].legend()
axes[2].set_title("State of Charge")

fig.suptitle("SPMe (monolithic) vs DFN (co-simulation) — 1 C Discharge", fontsize=12)
plt.tight_layout()
plt.show()


## Part 2: DFN with External Thermal Model

Same feedback topology as notebook 02, but with `CellCoSimElectrical` instead of
`CellElectrical`. We compare two macro-step sizes to illustrate ZOH approximation error.


In [ ]:
mass = 0.065  # [kg]
Cp   = 750.0  # [J kg⁻¹ K⁻¹]
UA   = 0.5    # [W K⁻¹]


def run_cosim_external(dt_macro):
    I_src_cs   = Constant(I_1C)
    T_src_cs   = Constant(T_amb0)
    cell_cs    = CellCoSimElectrical(
        model=pybamm.lithium_ion.DFN(options={
            "thermal": "isothermal",
            "calculate heat source for isothermal models": "true",
        }),
        initial_soc=1.0,
        dt=dt_macro,
    )
    thermal_cs = LumpedThermal(mass=mass, Cp=Cp, UA=UA, T0=T_amb0)
    sco        = Scope(labels=["V", "SOC", "T_cell"])

    sim = Simulation(
        blocks=[I_src_cs, T_src_cs, cell_cs, thermal_cs, sco],
        connections=[
            Connection(I_src_cs,          cell_cs["I"]),
            Connection(thermal_cs["T"],   cell_cs["T_cell"]),
            Connection(cell_cs["Q_dot"],  thermal_cs["Q_dot"]),
            Connection(T_src_cs,          thermal_cs["T_amb"]),
            Connection(cell_cs["V"],      sco[0]),
            Connection(cell_cs["SOC"],    sco[1]),
            Connection(thermal_cs["T"],   sco[2]),
        ],
        dt=dt_macro,
        Solver=ESDIRK43,
    )
    sim.run(t_end)
    t, [V, SOC, T_cell] = sco.read()
    return t, V, T_cell, SOC


In [ ]:
print("Running DFN co-sim with external thermal (dt = 30 s)...")
t_30, V_30, T_30, SOC_30 = run_cosim_external(dt_macro=30.0)

print("Running DFN co-sim with external thermal (dt =  5 s)...")
t_5,  V_5,  T_5,  SOC_5  = run_cosim_external(dt_macro=5.0)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].plot(t_30[1:] / 3600, V_30[1:],       label="dt = 30 s",  color="darkorange")
axes[0].plot(t_5[1:]  / 3600, V_5[1:],  "--", label="dt =  5 s",  color="steelblue")
axes[0].set_xlabel("Time / h")
axes[0].set_ylabel("Terminal voltage / V")
axes[0].set_ylim(3.0, 4.3)
axes[0].legend()
axes[0].set_title("Voltage")

axes[1].plot(t_30[1:] / 3600, T_30[1:] - 273.15,       label="dt = 30 s",  color="darkorange")
axes[1].plot(t_5[1:]  / 3600, T_5[1:]  - 273.15, "--", label="dt =  5 s",  color="steelblue")
axes[1].set_xlabel("Time / h")
axes[1].set_ylabel("Cell temperature / °C")
axes[1].legend()
axes[1].set_title("Temperature")

axes[2].plot(t_30[1:] / 3600, SOC_30[1:] * 100,       label="dt = 30 s",  color="darkorange")
axes[2].plot(t_5[1:]  / 3600, SOC_5[1:]  * 100, "--", label="dt =  5 s",  color="steelblue")
axes[2].set_xlabel("Time / h")
axes[2].set_ylabel("SOC / %")
axes[2].set_ylim(0, 105)
axes[2].legend()
axes[2].set_title("State of Charge")

fig.suptitle(
    "DFN Co-Simulation + External Thermal — Effect of Macro-Step Size",
    fontsize=12,
)
plt.tight_layout()
plt.show()


## Summary

| Block | Model | Thermal |
|-------|-------|---------|
| `CellElectrothermal` | SPMe / SPM (ODE) | Built-in |
| `CellCoSimElectrothermal` | Any incl. DFN (DAE) | Built-in |
| `CellElectrical` | SPMe / SPM (ODE) | External (`LumpedThermal`) |
| `CellCoSimElectrical` | Any incl. DFN (DAE) | External (`LumpedThermal`) |

- DFN resolves spatial gradients in both phases but produces a DAE — use the co-simulation blocks.
- A smaller macro-step `dt` reduces ZOH error at the cost of more PyBaMM `step()` calls.
- The external thermal loop extends naturally to multi-cell pack models.
